In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsinhueproject";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore-project`
URL 'abfss://metastore-project@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para el metastore-project del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_project_ssm CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_project_ssm
MANAGED LOCATION 'abfss://metastore-project@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente dev del proyecto';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_project_ssm.raw;
DROP SCHEMA IF EXISTS catalog_project_ssm.bronze;
DROP SCHEMA IF EXISTS catalog_project_ssm.silver;
DROP SCHEMA IF EXISTS catalog_project_ssm.golden;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_project_ssm.raw;
CREATE SCHEMA IF NOT EXISTS catalog_project_ssm.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_project_ssm.silver;
CREATE SCHEMA IF NOT EXISTS catalog_project_ssm.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_project_ssm.bronze.renovacion_prestamo_clientes_info_operacional (
    ID INTEGER,
    CLIENTE INTEGER,
    MES INTEGER,
    EDAD INTEGER,
    SEXO STRING,
    EST_CIVIL STRING,
    REGION STRING,
    Flag_LimProv INTEGER,
    ANTIGUEDAD_MES INTEGER,
    Meses_oferta INTEGER,
    FLAG_VENTA INTEGER
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/renovacion_prestamo_clientes_info_operacional"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_project_ssm.bronze.renovacion_prestamo_clientes_score_financiero (
    ID INTEGER,
    Linea_Renovado INTEGER,
    Plazo_Renovado INTEGER,
    Uso_Linea DOUBLE,
    Uso_TrimLinea DOUBLE,
    Nro_Entidades INTEGER,
    Dif_Entidades INTEGER,
    Saldo_Consumo DOUBLE,
    Ahorro INTEGER,
    Prestamo_vigente INTEGER,
    Promed_6Mdeuda DOUBLE,
    SUELDO_ESTIMADO DOUBLE,
    Deuda_Cubierta DOUBLE
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/renovacion_prestamo_clientes_score_financiero"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_project_ssm.silver.renovacion_prestamo_transformed (
    EDAD INTEGER,
    Flag_LimProv INTEGER,
    Meses_oferta DOUBLE,
    FLAG_VENTA INTEGER,
    Plazo_Renovado INTEGER,
    Nro_Entidades INTEGER,
    Dif_Entidades INTEGER,
    Uso_Linea_LOG DOUBLE,
    Uso_TrimLinea_LOG DOUBLE,
    Saldo_Consumo_LOG DOUBLE,
    SUELDO_ESTIMADO_LOG DOUBLE,
    ANTIGUEDAD_MES_LOG DOUBLE,
    Linea_Renovado_LOG DOUBLE,
    Ahorro_LOG DOUBLE,
    Prestamo_vigente_LOG DOUBLE,
    Promed_6Mdeuda_LOG DOUBLE,
    Deuda_Cubierta_LOG DOUBLE,
    REGION_CALLAO INTEGER,
    REGION_CENTRO INTEGER,
    REGION_LIMA_BALNEARIO INTEGER,
    REGION_LIMA_CENTRO INTEGER,
    REGION_LIMA_ESTE INTEGER,
    REGION_LIMA_MODERNA INTEGER,
    REGION_LIMA_NORTE INTEGER,
    REGION_LIMA_PROVINCIA INTEGER,
    REGION_LIMA_SUR INTEGER,
    REGION_NORTE INTEGER,
    REGION_OESTE INTEGER,
    REGION_ORIENTE INTEGER,
    REGION_SIERRA_CENTRAL INTEGER,
    REGION_SUR INTEGER,
    SEXO_F INTEGER,
    SEXO_M INTEGER,
    EST_CIVIL_C INTEGER,
    EST_CIVIL_D INTEGER,
    EST_CIVIL_S INTEGER,
    EST_CIVIL_U INTEGER,
    EST_CIVIL_V INTEGER,
    EST_CIVIL_X INTEGER,
    EST_CIVIL_Y INTEGER,
    Cluster INTEGER
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/renovacion_prestamo_transformed"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_project_ssm.golden.golden_renovacion_prestamo (
    FLAG_VENTA INTEGER COMMENT 'Indicador binario de éxito de venta (1 = Aceptó, 0 = Rechazó)',
    conteo_total_casos INTEGER COMMENT 'Cantidad total de clientes evaluados en el grupo',
    proporcion_mujeres DOUBLE COMMENT 'Proporción de clientes mujeres (0.00 a 1.00)',
    proporcion_hombres DOUBLE COMMENT 'Proporción de clientes hombres (0.00 a 1.00)',
    edad_promedio DOUBLE COMMENT 'Edad promedio de los clientes',
    edad_minima INTEGER COMMENT 'Edad mínima registrada en el segmento',
    edad_maxima INTEGER COMMENT 'Edad máxima registrada en el segmento',
    plazo_promedio_meses DOUBLE COMMENT 'Plazo promedio en meses elegido/ofertado',
    meses_oferta_promedio DOUBLE COMMENT 'Promedio de meses que la oferta estuvo activa',
    promedio_sueldo_log DOUBLE COMMENT 'Promedio del logaritmo del sueldo estimado',
    promedio_linea_ofertada_log DOUBLE COMMENT 'Promedio del logaritmo de la línea de crédito renovada',
    promedio_deuda_6m_log DOUBLE COMMENT 'Promedio del logaritmo de la deuda en los últimos 6 meses',
    promedio_antiguedad_log DOUBLE COMMENT 'Promedio del logaritmo de la antigüedad en meses del cliente'
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_renovacion_prestamo"
